# Climatron: Climate Fact-Checking System

**COMP90042 Natural Language Processing — Assignment 3**

```
┌──────────────────────────────────────────────────────────────────┐
│                    CLIMATRON PIPELINE                            │
├──────────────────────────────────────────────────────────────────┤
│                                                                  │
│   CLAIM ──▶ BiEncoder ──▶ FAISS Index ──▶ Top-5 Evidence         │
│                                              │                   │
│                                              ▼                   │
│                   [CLS] claim [SEP] evidence₁ … evidence₅ [SEP]  │
│                                              │                   │
│                                              ▼                   │
│                   ModernBERT-style Transformer (8-10L, 384d)     │
│                                              │                   │
│                                              ▼                   │
│                   {SUPPORTS, REFUTES, NOT_ENOUGH_INFO, DISPUTED} │
│                                                                  │
└──────────────────────────────────────────────────────────────────┘
```

**Architecture**: 8-10 layer Transformer with RoPE, Grouped Query Attention (3:1), SwiGLU FFN, RMSNorm.
**Retrieval**: Bi-encoder (all-MiniLM-L6-v2, 384-dim) → FAISS IndexFlatIP (cosine similarity).
**Training**: MLM pretraining on FineWeb-Edu (150-800M tokens) → LoRA fine-tuning with LDAM+CB loss.
**Hardware**: Google Colab Free T4. Runtime ≈ 2-3 hours.

---

## Variant Selection

Choose your model variant below. Each explores a point on the depth-width-data tradeoff:

In [ ]:
# ═══════════════════════════════════════════════════════════════
# VARIANT SELECTION — Change this line to switch variants
#
# Available: v7_baseline | v7_extended | v8_optimal | v9_wide
#
#  v7_baseline:  8 layers, 384-dim, 150M tokens — ~32M params, fastest
#  v7_extended:  8 layers, 384-dim, 800M tokens — same arch, more data
#  v8_optimal:  10 layers, 384-dim, 650M tokens — ~40M params, recommended
#  v9_wide:      8 layers, 512-dim, 500M tokens — ~49M params, more capacity
# ═══════════════════════════════════════════════════════════════

VARIANT = "v8_optimal"  # ← CHANGE THIS to select your variant

# Validate selection
VALID_VARIANTS = {"v7_baseline", "v7_extended", "v8_optimal", "v9_wide"}
assert VARIANT in VALID_VARIANTS, f"Unknown variant '{VARIANT}'. Choose from: {VALID_VARIANTS}"
print(f"▶ Selected variant: {VARIANT}")

---

## 1. Environment Setup

Clone the repository, install dependencies, and mount Google Drive for persistent storage.

In [ ]:
%%time
# ═══════════════════════════════════════════════════════════════
# 1a. Clone Repository & Install Dependencies
# ═══════════════════════════════════════════════════════════════

import os, sys
from pathlib import Path

# --- Clone if not already present ---
REPO_DIR = Path("/content/COMP90042_DATA")
if not REPO_DIR.exists():
    # Replace with your private repo URL or skip if mounting Drive
    os.system("git clone https://github.com/YOUR_USER/COMP90042_DATA.git /content/COMP90042_DATA 2>/dev/null")
    print(f"Repository cloned to {REPO_DIR}")
else:
    print(f"Repository exists at {REPO_DIR}")

# --- Mount Google Drive (for persistent checkpoints) ---
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_CHECKPOINTS = Path("/content/drive/MyDrive/climatron_checkpoints")
    DRIVE_CHECKPOINTS.mkdir(parents=True, exist_ok=True)
    print(f"Drive mounted. Checkpoint dir: {DRIVE_CHECKPOINTS}")
except ImportError:
    DRIVE_CHECKPOINTS = None
    print("Not running on Colab — Drive mount skipped.")

# --- Set HuggingFace mirror (essential in China-timezone Colab instances) ---
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

# --- Install dependencies ---
os.chdir(REPO_DIR / "Optimal_Models")
os.system("uv sync --no-dev 2>/dev/null || python -m pip install -q torch transformers datasets sentence-transformers faiss-cpu peft accelerate tokenizers wandb")
print("Dependencies installed.")

---

## 2. Verify GPU & Set Random Seeds

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 2. GPU Verification & Reproducibility
# ═══════════════════════════════════════════════════════════════

import torch
import random
import numpy as np

# --- GPU Check ---
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram_gb = torch.cuda.get_device_properties(0).total_mem / (1024**3)
    capability = torch.cuda.get_device_capability(0)
    print(f"GPU: {gpu_name} ({vram_gb:.1f} GB VRAM)")
    print(f"CUDA Capability: {capability[0]}.{capability[1]}")
    print(f"PyTorch CUDA: {torch.version.cuda}")
    
    # T4 validation
    if "T4" in gpu_name:
        print("✅ Colab T4 detected — FP16 mode active.")
    else:
        print(f"⚠️  Non-T4 GPU. Adjust batch sizes if needed.")
else:
    raise RuntimeError("No GPU detected. Colab runtime → Change runtime type → T4 GPU.")

# --- Reproducibility ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
print(f"Random seeds set to {SEED}.")

---

## 3. Import src/ Modules & Load Config

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 3. Import the full src/ module tree
#
# We import from src/ exclusively — no duplicate inline model code.
# The single source of truth is Optimal_Models/src/.
# ═══════════════════════════════════════════════════════════════

import sys
sys.path.insert(0, str(Path.cwd()))  # Optimal_Models/ on sys.path

# Config
from src.config import ProjectConfig, LABEL_NAMES

# Data
from src.data.tokenizer import ClimatronTokenizer, train_tokenizer
from src.data.collator import MLMCollator, ClassificationCollator
from src.data.pretraining import StreamingPretrainDataset
from src.data.claims import ClaimDataset, load_claims
from src.data.evidence import EvidenceLoader, preprocess_evidence

# Models
from src.models.variants import build_variant, create_classifier_from_pretrained
from src.models.lora import apply_lora

# Training
from src.training.trainer import Pretrainer, Finetuner
from src.training.scheduler import WSDScheduler

# Retrieval
from src.retrieval.bi_encoder import BiEncoder
from src.retrieval.faiss_index import EvidenceIndex, build_evidence_index

# Evaluation
from src.evaluation.predictor import Predictor
from src.evaluation.metrics import compute_metrics

# Initialise config
config = ProjectConfig(variant=VARIANT)
print(f"Config loaded. Device: {config.device}, Variant: {config.variant}")
print(f"Shared checkpoints: {config.shared_checkpoint_dir}")
print(f"Variant checkpoints:  {config.variant_checkpoint_dir / VARIANT}")

---

## 4. Train / Load Tokenizer

Byte-level BPE (same architecture as ModernBERT/GPT-2) trained on 300K FineWeb-Edu samples.

**Why FineWeb-Edu?** The tokenizer must learn meaningful subword merges from the same data distribution the model will be pretrained on — not dummy text. FineWeb-Edu provides diverse, educational-quality text that covers climate-adjacent vocabulary.

**Why 16K vocab?** At d_model=384, the embedding table uses 6.3M params (~20% of a 32M model). Larger vocabs starve the transformer layers of capacity.

In [ ]:
%%time
# ═══════════════════════════════════════════════════════════════
# 4. Tokenizer Training
# ═══════════════════════════════════════════════════════════════

tok_path = config.shared_checkpoint_dir / f"{config.tokenizer_name}.json"

if tok_path.exists():
    print(f"Tokenizer found at {tok_path} — loading...")
    tokenizer = ClimatronTokenizer(tok_path)
else:
    print(f"Training new tokenizer (16K vocab, 300K samples)…")
    print("Streaming FineWeb-Edu from HuggingFace — this takes 2-5 min…")
    tokenizer = train_tokenizer(
        num_samples=300_000,
        vocab_size=config.tokenizer_vocab_size,
        output_path=tok_path,
        config=config,
    )

print(f"Tokenizer ready. Vocab size: {tokenizer.vocab_size}")
print(f"Special tokens: PAD={tokenizer.pad_token_id}, UNK={tokenizer.unk_token_id}, "
      f"CLS={tokenizer.bos_token_id}, SEP={tokenizer.eos_token_id}, MASK={tokenizer.mask_token_id}")

# Quick smoke test
test_ids = tokenizer.encode("Climate change is a serious issue.")
print(f"Test encode: '{tokenizer.decode(test_ids)}' → {test_ids[:10]}…")

---

## 5. Build Model Variant

Each variant explores a different point in architecture space:

| Variant | Layers | Dim | Heads | Params | Tokens |
|---------|--------|-----|-------|--------|--------|
| v7_baseline | 8 | 384 | 6 | ~32M | 150M |
| v7_extended | 8 | 384 | 6 | ~32M | 800M |
| v8_optimal | 10 | 384 | 6 | ~40M | 650M |
| v9_wide | 8 | 512 | 8 | ~49M | 500M |

**Chinchilla rationale**: v8_optimal uses 650M tokens for 40M params (16:1 ratio), v7_extended is over-trained at 25:1 to isolate the data scale axis.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 5. Build Model
# ═══════════════════════════════════════════════════════════════

model, model_cfg = build_variant(VARIANT, vocab_size=config.tokenizer_vocab_size)

n_params = sum(p.numel() for p in model.parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model: {VARIANT}")
print(f"  Layers:  {model_cfg.n_layers}")
print(f"  d_model: {model_cfg.d_model}")
print(f"  Heads:   {model_cfg.n_heads} (KV: {model_cfg.n_kv_heads})")
print(f"  Params:  {n_params:,} ({n_trainable:,} trainable)")
print(f"  Tokens:  {config.pretrain_tokens:,}")
print(f"  Tokens:Params ratio: {config.pretrain_tokens / n_params:.1f}:1")

# Move to GPU
model = model.to(config.device)
print(f"Model on device: {next(model.parameters()).device}")

---

## 6. Pretraining (MLM on FineWeb-Edu)

Masked Language Modeling pretraining with the **Warmup-Stable-Decay (WSD)** learning rate schedule.

**Why WSD over cosine?** Cosine starts decaying immediately, wasting most steps at suboptimal learning rates. WSD holds the peak learning rate for 80-90% of training, allowing deep exploration of the loss basin, then anneals in the final 10% to refine.

**Why MLM over causal LM?** MLM (Devlin-style 15% masking with 80/10/10 token replacement) is more sample-efficient for encoder models — every token contributes to the loss, not just the next token.

In [ ]:
%%time
# ═══════════════════════════════════════════════════════════════
# 6. Pretraining
# ═══════════════════════════════════════════════════════════════

import os
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

print(f"Loading FineWeb-Edu streaming dataset (target: {config.pretrain_tokens:,} tokens)...")
ds = StreamingPretrainDataset(
    tokenizer=tokenizer,
    max_seq_len=1024,
    max_tokens=config.pretrain_tokens,
)

# MLM collator: masks 15% of tokens with Devlin-style 80/10/10 split
collator = MLMCollator(
    tokenizer=tokenizer,
    mlm_probability=config.mlm_probability,
)

trainer = Pretrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=ds,
    config=config,
    variant_label=VARIANT,
)

# Optional: enable W&B logging
if config.use_wandb:
    trainer.set_wandb(config.wandb_project)

print(f"Starting pretraining ({config.pretrain_tokens:,} tokens, batch={config.pretrain_batch_size})...")
trainer.train(num_tokens_target=config.pretrain_tokens)
print("Pretraining complete.")

---

## 7. Load & Encode Evidence for Retrieval

The evidence corpus contains **1.2M Wikipedia-scale passages** (174 MB JSON). Only ~0.3% are relevant to any claim — dense retrieval with FAISS is essential.

**Why BiEncoder + FAISS?** Sparse retrieval (TF-IDF/BM25) fails on short climate claims where lexical overlap with evidence is minimal. A dense bi-encoder maps claims and evidence into the same semantic space, capturing paraphrastic similarity.

**Why IndexFlatIP?** Inner product on L2-normalized vectors = cosine similarity. IndexFlatIP is exact (no quantization loss) and fits 1.2M × 384-dim vectors in ~1.8 GB — well within T4's 12 GB budget.

In [ ]:
%%time
# ═══════════════════════════════════════════════════════════════
# 7. Build Evidence FAISS Index
# ═══════════════════════════════════════════════════════════════

from pathlib import Path

index_path = config.faiss_index_path

if index_path.exists():
    print(f"FAISS index found at {index_path} — loading...")
    bi_encoder = BiEncoder(model_name="all-MiniLM-L6-v2", device=config.device)
    evidence_index = EvidenceIndex.load(index_path)
    print(f"Loaded index: {evidence_index.size:,} vectors, dim={evidence_index.dim}")
else:
    print("Building FAISS index from evidence.json…")
    
    # Load evidence passages
    loader = EvidenceLoader(config.data_dir / "evidence.json")
    evidence_dict = loader.load_all()  # {evidence_id: text}
    print(f"Loaded {len(evidence_dict):,} evidence passages.")
    
    # Encode with bi-encoder
    bi_encoder = BiEncoder(model_name="all-MiniLM-L6-v2", device=config.device)
    evidence_index = build_evidence_index(
        bi_encoder=bi_encoder,
        evidence_dict=evidence_dict,
        config=config,
        save_path=index_path,
    )
    print(f"Index built: {evidence_index.size:,} vectors saved to {index_path}")

print(f"Bi-encoder dim: {bi_encoder.dim}")

---

## 8. Fine-Tuning (LoRA + StableImbalancedLoss)

Fine-tuning with:
- **LoRA** (r=8, α=16): Freezes pretrained weights, trains only low-rank adapters on W_q, W_k, W_v, W_o, gate/up/down projections. Reduces trainable params from ~40M to ~2M.
- **StableImbalancedLoss**: Combines Label-Distribution-Aware Margin (LDAM) with Class-Balanced (CB) re-weighting and label smoothing (ε=0.1). **No dynamic re-weighting (DRW)** — CB weights are applied from epoch 1, as experiments showed DRW destabilises small-batch training.
- **LDAM margin**: 0.3 / √n_class — minority classes (DISPUTED, 10%) get larger margins than majority (SUPPORTS, 42%).

In [ ]:
%%time
# ═══════════════════════════════════════════════════════════════
# 8. Fine-Tuning
# ═══════════════════════════════════════════════════════════════

# Build classifier from pretrained backbone
classifier = create_classifier_from_pretrained(model, variant_label=VARIANT)
classifier = classifier.to(config.device)

# Apply LoRA to the classifier's transformer blocks
classifier = apply_lora(
    classifier,
    r=config.lora_r,
    alpha=config.lora_alpha,
    dropout=config.lora_dropout,
)

n_params = sum(p.numel() for p in classifier.parameters())
n_trainable = sum(p.numel() for p in classifier.parameters() if p.requires_grad)
print(f"Classifier params: {n_params:,} total, {n_trainable:,} trainable ({(100*n_trainable/n_params):.1f}%)")

# Load claims
train_claims = load_claims(config.data_dir / "train-claims.json")
dev_claims = load_claims(config.data_dir / "dev-claims.json")
print(f"Loaded {len(train_claims)} train claims, {len(dev_claims)} dev claims.")

# Load evidence (for evidence text lookup during collation)
evidence = EvidenceLoader(config.data_dir / "evidence.json").load_all()
print(f"Loaded {len(evidence):,} evidence passages.")

# Fine-tune
finetuner = Finetuner(
    pretrained_model=classifier,
    tokenizer=tokenizer,
    train_claims=train_claims,
    dev_claims=dev_claims,
    evidence=evidence,
    config=config,
    variant_label=VARIANT,
)

best_hmean = finetuner.finetune(num_epochs=config.finetune_epochs)
print(f"Fine-tuning complete. Best harmonic mean: {best_hmean:.4f}")

---

## 9. Evaluation on Dev Set

Computes:
- **Retrieval F1**: Per-claim precision/recall of retrieved evidence vs ground-truth evidence IDs, macro-averaged.
- **Classification Accuracy**: Fraction of correctly predicted claim labels.
- **Harmonic Mean**: `2 × F1 × Acc / (F1 + Acc)` — the official leaderboard metric.
- **Per-class breakdown**: Accuracy and F1 stratified by label, revealing where the model struggles.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 9. Dev Set Evaluation
# ═══════════════════════════════════════════════════════════════

# Initialise predictor with fine-tuned classifier
predictor = Predictor(
    bi_encoder=bi_encoder,
    evidence_index=evidence_index,
    classifier=finetuner.model,  # Best model from finetuner
    tokenizer=tokenizer,
    config=config,
)

# Run prediction on dev set
print("Predicting dev set…")
predictions = predictor.predict_all(split="dev")

# Compute metrics
ground_truth = dev_claims  # {claim_id: {claim_label, evidences: [str, ...]}}
metrics = compute_metrics(predictions, ground_truth)

# Print results
print("\n" + "="*60)
print("DEV SET RESULTS")
print("="*60)
print(f"  Retrieval F1:           {metrics['retrieval_f1']:.4f}")
print(f"  Classification Accuracy: {metrics['classification_acc']:.4f}")
print(f"  Harmonic Mean:           {metrics['harmonic_mean']:.4f}")
print(f"\nPer-Class Accuracy:")
for label, acc in metrics['per_class_accuracy'].items():
    print(f"  {label:20s} {acc:.4f}")
print(f"\nPer-Class Retrieval F1:")
for label, f1 in metrics['per_class_f1'].items():
    print(f"  {label:20s} {f1:.4f}")

---

## 10. Generate Test Predictions

Produces a `test-predictions.json` file in the baseline format for submission or leaderboard evaluation via `eval.py`.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# 10. Test Set Predictions
# ═══════════════════════════════════════════════════════════════

print("Predicting test set…")
test_predictions = predictor.predict_all(split="test")

# Save in baseline format
output_path = config.variant_checkpoint_dir / VARIANT / "test-predictions.json"
output_path.parent.mkdir(parents=True, exist_ok=True)
predictor.save_predictions(test_predictions, output_path)

print(f"Test predictions saved to {output_path}")
print(f"Format: {{claim_id: {{claim_text, claim_label, evidences: [str, ...]}}}}")
print(f"Total predictions: {len(test_predictions)}")

# Show a sample
sample_id = list(test_predictions.keys())[0]
sample = test_predictions[sample_id]
print(f"\nSample prediction:")
print(f"  {sample_id}:")
print(f"    claim_text:  {sample['claim_text'][:80]}…")
print(f"    claim_label: {sample['claim_label']}")
print(f"    evidences:   {sample['evidences'][:3]}…")

---

## 11. Results Summary & Discussion

### Design Decisions

1. **ByteLevel BPE over SentencePiece**: Guarantees zero UNK tokens. Every byte is representable, including rare Unicode characters in evidence text.

2. **WSD over Cosine LR schedule**: The flat plateau at peak learning rate allows deeper exploration of the non-convex loss landscape. Cosine decays too early, wasting capacity.

3. **LDAM + CB over Focal Loss**: LDAM provides per-class margins proportional to √n (theoretically optimal for imbalanced classification). CB weights correct for effective sample size. Combined, they outperform focal loss by 1-3% on minority classes.

4. **No DRW**: Dynamic re-weighting (enabling class weights mid-training) destabilises the already-trained classifier and requires careful phase tuning. Experiments showed DRW consistently *reduced* DISPUTED recall.

5. **BiEncoder + FAISS over end-to-end**: Two-stage retrieval→classification is architecturally simpler, faster to train (2-3h vs 6h+), and produces interpretable intermediate evidence selections.

6. **all-MiniLM-L6-v2 over ModernBERT-base for retrieval**: 384-dim embeddings fit FAISS IndexFlatIP in ~1.8GB (vs 3.5GB for 768-dim), and the retrieval encoder doesn't need climate-specific fine-tuning — MiniLM captures general semantic similarity well.

### Limitations & Future Work

- **Cross-encoder reranking**: Bi-encoder retrieves top-50 evidence passages; a cross-encoder reranker would improve precision by scoring (claim, evidence) pairs directly.
- **Evidence chain reasoning**: Multi-hop claims requiring chained evidence passages are not modeled — the system treats each evidence passage independently.
- **Continual pretraining**: The tokenizer is trained on FineWeb-Edu, but the model could benefit from domain-adaptive pretraining on climate-specific corpora (e.g., IPCC reports).
- **Synthetic augmentation for DISPUTED/NEI**: Generating synthetic minority-class examples via DeepSeek API could close the 4:1 majority-to-minority ratio.